# 🏠 Voursa.com Real Estate Scraper v3

Ce notebook scrape les annonces immobilières de **voursa.com** en deux phases :
- **Phase 1** : Collecte des URLs par sous-catégorie (scroll + "View More")
- **Phase 2** : Extraction des détails de chaque annonce

> **Prérequis** : Google Chrome installé sur la machine.

## 📦 Installation des dépendances

In [ ]:
!pip install selenium pandas webdriver-manager

## 📚 Imports

In [ ]:
import sys, io, time, re, logging, random
from datetime import datetime
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, StaleElementReferenceException,
    ElementClickInterceptedException, ElementNotInteractableException,
)

try:
    from webdriver_manager.chrome import ChromeDriverManager
    USE_WDM = True
except ImportError:
    USE_WDM = False

print("✅ Imports OK | webdriver-manager:", USE_WDM)

## ⚙️ Configuration du logging

In [ ]:
# UTF-8 safe logging on Windows
_stream = io.TextIOWrapper(sys.stdout.buffer, encoding="utf-8", errors="replace") \
    if hasattr(sys.stdout, "buffer") else sys.stdout

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("voursa_scraper.log", encoding="utf-8"),
        logging.StreamHandler(_stream),
    ],
)
log = logging.getLogger(__name__)
print("✅ Logging configuré")

## 🔧 Paramètres de configuration

Modifiez ces valeurs selon vos besoins.

In [ ]:
BASE          = "https://voursa.com/EN"
REAL_ESTATE   = f"{BASE}/categories/real_estate"
TARGET_ROWS   = 4000          # Nombre d'annonces cibles
OUTPUT_CSV    = "voursa_real_estate.csv"
CHECKPOINT    = "voursa_checkpoint.csv"
URLS_CACHE    = "voursa_urls_cache.txt"
RELOAD_EVERY  = 8             # Reset DOM après N clics "View More"
PAGE_WAIT     = 20            # Secondes max d'attente JS
SCROLL_SLEEP  = 1.5           # Pause après chaque scroll
CLICK_SLEEP   = (0.4, 0.9)
DETAIL_SLEEP  = (1.5, 3.0)
NO_NEW_LIMIT  = 4             # Rounds vides consécutifs avant de passer à la suite

# Sous-catégories immobilières de voursa.com
SUBCATEGORIES = [
    ("Residential Property",      1),
    ("Land Plot",                 2),
    ("Office",                    3),
    ("Warehouse",                 4),
    ("Store",                     5),
    ("Commercial and Industrial", 6),
    ("Other Real Estate",         7),
]

print("✅ Config chargée")
print(f"   Cible : {TARGET_ROWS} annonces → {OUTPUT_CSV}")

## 🗺️ Bases de connaissances (villes, quartiers, types)

In [ ]:
NOUAKCHOTT_QUARTERS = [
    "tevragh zeina","teyarett","ksar","el mina","arafat",
    "dar naim","toujounine","sebkha","riadh","capital",
    "socogim","pk","ilot","lotissement","bouhdida",
    "tfareg zeina","tfargh zeina","tvz","secteur","bloc","block","blok",
]
MAURITANIAN_CITIES = [
    "nouakchott","nouadhibou","rosso","kaedi","zouerate",
    "kiffa","tidjikja","atar","nema","selibaby","aleg",
    "boutilimit","akjoujt","bassikounou",
]
TYPE_BIEN_FR = {
    "villa":["villa"],"terrain":["terrain","lot","land","plot"],
    "duplex":["duplex"],"appartement":["appartement","apartment","appart"],
    "maison":["maison","house"],"bureau":["bureau","office"],
    "entrepot":["entrepot","warehouse"],"magasin":["magasin","store"],
    "immeuble":["immeuble","building"],
}
TYPE_BIEN_AR = {
    "villa":["فيلا","فلا"],
    "terrain":["نيمرو","ارض","قطعة","تراب","شانتيه"],
    "duplex":["ديبلكس","جبلكس"],
    "appartement":["شقة","شقه"],
    "maison":["دار","منزل","بيت"],
    "bureau":["مكتب"],
    "magasin":["محل","متجر","دكان"],
    "immeuble":["عمارة","بناية"],
}
SALE_AR    = ["للبيع","بيع"]
RENT_AR    = ["للايجار","كراء","ايجار"]
EXTRA_FEAT = ["piscine","pool","garage","jardin","garden","terrasse","terrace",
              "climatisation","clim","meuble","furnished","parking",
              "ascenseur","elevator","gardien","neuf","nouveau","new"]

print("✅ Bases de connaissances chargées")

## 🛠️ Fonctions utilitaires

In [ ]:
def rand_sleep(lo=0.4, hi=0.9):
    time.sleep(random.uniform(lo, hi))

def infer_type_bien(text):
    t = text.lower()
    for label, kws in TYPE_BIEN_FR.items():
        if any(kw in t for kw in kws):
            return label
    for label, kws in TYPE_BIEN_AR.items():
        if any(kw in text for kw in kws):
            return label
    return "autre"

def infer_type_annonce(text):
    t = text.lower()
    if any(kw in t for kw in ["for sale","vente","sale","a vendre"]) \
       or any(kw in text for kw in SALE_AR):
        return "Vente"
    if any(kw in t for kw in ["for rent","location","rent","louer","to let"]) \
       or any(kw in text for kw in RENT_AR):
        return "Location"
    return "Inconnu"

def extract_quartier(text):
    t = text.lower()
    for q in NOUAKCHOTT_QUARTERS:
        if q in t:
            return q.title()
    return ""

def extract_ville(text):
    t = text.lower()
    for city in MAURITANIAN_CITIES:
        if city in t:
            return city.title()
    return "Nouakchott"

print("✅ Fonctions utilitaires définies")

## 🌐 Initialisation du navigateur Chrome

In [ ]:
def build_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    if USE_WDM:
        svc = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=svc, options=opts)
    else:
        driver = webdriver.Chrome(options=opts)
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"},
    )
    return driver

print("✅ Fonction build_driver() définie")
print("   Astuce : passez headless=False pour voir le navigateur en action")

## 📜 Helpers de scroll et clic

In [ ]:
def full_scroll(driver):
    """Scroll lent vers le bas pour déclencher le lazy loading."""
    total_h = driver.execute_script("return document.body.scrollHeight")
    step = max(600, total_h // 8)
    pos = 0
    while pos < total_h:
        pos = min(pos + step, total_h)
        driver.execute_script(f"window.scrollTo(0, {pos})")
        time.sleep(0.3)
    time.sleep(SCROLL_SLEEP)


def click_view_more(driver) -> bool:
    """Cherche et clique sur le bouton 'View More'. Retourne True si cliqué."""
    selectors = [
        "//button[contains(translate(normalize-space(.),"
        "'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'VIEW MORE')]",
        "//button[contains(translate(normalize-space(.),"
        "'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'LOAD MORE')]",
        "//button[contains(translate(normalize-space(.),"
        "'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'SHOW MORE')]",
        "//a[contains(translate(normalize-space(.),"
        "'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'VIEW MORE')]",
        "//a[contains(translate(normalize-space(.),"
        "'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ'),'VIEW MORE')]",
        "//button[contains(@class,'more') or contains(@class,'More')]",
        "//button[contains(@class,'load') or contains(@class,'Load')]",
        "//*[@data-testid='load-more']",
        "//*[@data-testid='view-more']",
    ]
    for sel in selectors:
        try:
            btn = driver.find_element(By.XPATH, sel)
            driver.execute_script("arguments[0].scrollIntoView({block:'center'})", btn)
            time.sleep(0.3)
            try:
                btn.click()
            except (ElementClickInterceptedException, ElementNotInteractableException):
                driver.execute_script("arguments[0].click()", btn)
            rand_sleep(*CLICK_SLEEP)
            return True
        except NoSuchElementException:
            continue
        except Exception:
            continue
    return False


def wait_for_cards(driver, timeout=PAGE_WAIT):
    """Attend que les cartes d'annonces apparaissent dans le DOM."""
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR, "a[href*='/ads/']")) > 0
        )
        time.sleep(1.5)
        return True
    except TimeoutException:
        return False

print("✅ Helpers scroll/clic définis")

## 🔍 Phase 1 : Collecte des URLs

In [ ]:
def collect_urls(driver, target: int) -> list:
    """
    Parcourt chaque sous-catégorie, scrolle et clique sur View More.
    Retourne une liste de dicts {url, subcategory}.
    """
    all_items: dict = {}  # url -> {url, subcategory}

    for sub_name, sub_id in SUBCATEGORIES:
        if len(all_items) >= target + 500:
            break

        page_url = f"{REAL_ESTATE}?subcategoryId={sub_id}"
        log.info(f"[Phase 1] Sous-catégorie : {sub_name}  (id={sub_id})")

        driver.get(page_url)
        if not wait_for_cards(driver):
            log.warning(f"  Aucune carte pour {sub_name}, passage à la suivante.")
            continue

        click_count   = 0
        no_new_streak = 0

        while True:
            cards = driver.find_elements(By.CSS_SELECTOR, "a[href*='/ads/']")
            added = 0
            for c in cards:
                try:
                    href = c.get_attribute("href") or ""
                    if href and "/ads/" in href and href not in all_items:
                        all_items[href] = {"url": href, "subcategory": sub_name}
                        added += 1
                except StaleElementReferenceException:
                    continue

            print(f"\r  {sub_name}: {len(all_items)} URLs totales  "
                  f"(+{added} ce round, clics={click_count})", end="", flush=True)

            if added == 0:
                no_new_streak += 1
                if no_new_streak >= NO_NEW_LIMIT:
                    print()
                    log.info(f"  Aucune nouvelle URL depuis {NO_NEW_LIMIT} rounds → passage à la suite")
                    break
            else:
                no_new_streak = 0

            full_scroll(driver)
            clicked = click_view_more(driver)
            if not clicked:
                full_scroll(driver)
                clicked = click_view_more(driver)
                if not clicked:
                    print()
                    log.info(f"  Bouton View More introuvable → passage à la suite")
                    break

            click_count += 1

            if click_count % RELOAD_EVERY == 0:
                print()
                log.info(f"  Rechargement de la page pour réinitialiser le DOM (clic #{click_count}) ...")
                driver.get(page_url)
                time.sleep(2.5)
                wait_for_cards(driver)
                for _ in range(min(click_count, 30)):
                    full_scroll(driver)
                    if not click_view_more(driver):
                        break
                    rand_sleep(0.3, 0.6)

            rand_sleep(*CLICK_SLEEP)

        log.info(f"  => {sub_name} terminé. Pool : {len(all_items)} URLs")

    items = list(all_items.values())

    with open(URLS_CACHE, "w", encoding="utf-8") as f:
        for it in items:
            f.write(f"{it['url']}\t{it['subcategory']}\n")
    log.info(f"[Phase 1] Terminé. {len(items)} URLs uniques sauvegardées dans {URLS_CACHE}")
    return items

print("✅ Fonction collect_urls() définie")

## 📄 Phase 2 : Extraction des détails

In [ ]:
def extract_attrs_from_source(page_source, body_text):
    attrs = {}
    pat = re.compile(
        r'(Lot\s*Size|Rooms?|Bathrooms?|Salons?|Closest\s*Landmark|'
        r'Surface|Floor|Type|Living\s*Room|Product\s*Type)\s*[*·•]\s*([^\n]{1,100})',
        re.IGNORECASE,
    )
    for m in pat.finditer(body_text):
        k = re.sub(r'\s+', ' ', m.group(1)).strip().lower()
        v = m.group(2).strip()
        if k not in attrs:
            attrs[k] = v
    html_pat = re.compile(
        r'(Lot\s*Size|Rooms?|Bathrooms?|Salons?|Closest\s*Landmark|'
        r'Surface|Floor|Living\s*Room)[^<"]{0,40}'
        r'(?:>|[·•]|\s[-:]\s)([\d,\.]+(?:\s*m[2²])?|[A-Za-z][^<"]{1,60})',
        re.IGNORECASE,
    )
    for m in html_pat.finditer(page_source):
        k = re.sub(r'\s+', ' ', m.group(1)).strip().lower()
        v = m.group(2).strip()
        if k not in attrs and len(v) < 80:
            attrs[k] = v
    return attrs


def scrape_detail(driver, url, subcategory=""):
    """Visite une page d'annonce et extrait tous les champs."""
    try:
        driver.get(url)
        try:
            WebDriverWait(driver, PAGE_WAIT).until(
                lambda d: (
                    len(d.find_elements(By.TAG_NAME, "h1")) > 0
                    and d.find_element(By.TAG_NAME, "h1").text.strip() != ""
                )
            )
        except TimeoutException:
            try:
                WebDriverWait(driver, 8).until(lambda d: "MRU" in d.page_source)
            except TimeoutException:
                log.warning(f"  Timeout: {url[:80]}")
                return None
        time.sleep(random.uniform(*DETAIL_SLEEP))
    except Exception as e:
        log.warning(f"  Erreur lors du chargement de {url[:80]}: {e}")
        return None

    page_source = driver.page_source
    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text
    except Exception:
        body_text = ""

    # Titre
    titre = ""
    try:
        titre = driver.find_element(By.TAG_NAME, "h1").text.strip()
    except NoSuchElementException:
        pass

    # Prix
    prix = ""
    m = re.search(r'([\d\s,]+)\s*MRU', page_source)
    if m:
        prix = m.group(0).strip()
    if not prix:
        m = re.search(r'([\d\s,]+)\s*MRU', body_text)
        if m:
            prix = m.group(0).strip()

    # Attributs
    attrs = extract_attrs_from_source(page_source, body_text)

    def ga(*keys):
        for k in keys:
            v = attrs.get(k.lower(), "")
            if v:
                return v
        return ""

    surface_m2  = ga("lot size", "surface", "superficie")
    nb_chambres = ga("rooms", "room", "chambres", "bedroom", "bedrooms")
    nb_salons   = ga("salon", "salons", "living room")
    nb_sdb      = ga("bathroom", "bathrooms", "sdb", "salle de bain")
    landmark    = ga("closest landmark", "landmark")

    # Description
    description = ""
    for sel in ["[class*='escription']","[class*='ontent']","[class*='ody']",
                "article p","main p","section p"]:
        try:
            for el in driver.find_elements(By.CSS_SELECTOR, sel):
                t = el.text.strip()
                if len(t) > 50 and t != titre:
                    description = t[:2000]
                    break
            if description:
                break
        except Exception:
            pass

    # Date
    date_pub = ""
    date_pats = [
        r'\d+\s*(?:second|minute|hour|day|week|month|year|heure|jour|semaine|mois|an)s?\s*ago',
        r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}',
    ]
    for pat in date_pats:
        m = re.search(pat, body_text, re.IGNORECASE)
        if m:
            date_pub = m.group(0)
            break

    # Caractéristiques
    full_text = f"{titre} {description} {landmark} {subcategory} {url}"
    t_low = full_text.lower()
    extras = [f for f in EXTRA_FEAT if f in t_low]
    skip = {"lot size","rooms","room","salon","salons","bathroom","bathrooms",
            "closest landmark","surface","superficie","chambres","bedroom",
            "bedrooms","sdb","salle de bain","living room"}
    rem = {k: v for k, v in attrs.items() if k not in skip}
    caracteristiques = "; ".join(extras)
    if rem:
        caracteristiques += (" | " if caracteristiques else "") + \
                            "; ".join(f"{k}: {v}" for k, v in rem.items())

    return {
        "titre":            titre,
        "type_bien":        infer_type_bien(full_text),
        "type_annonce":     infer_type_annonce(full_text),
        "prix":             prix,
        "surface_m2":       surface_m2,
        "nb_chambres":      nb_chambres,
        "nb_salons":        nb_salons,
        "nb_sdb":           nb_sdb,
        "quartier":         extract_quartier(full_text),
        "ville":            extract_ville(full_text),
        "description":      description,
        "source":           "voursa.com",
        "date_publication": date_pub,
        "caracteristiques": caracteristiques,
        "subcategory":      subcategory,
        "url":              url,
        "scraped_at":       datetime.utcnow().isoformat(),
    }

print("✅ Fonctions d'extraction définies")

## 💾 Sauvegarde des checkpoints

In [ ]:
def save_checkpoint(records, path):
    pd.DataFrame(records).to_csv(path, index=False, encoding="utf-8-sig")
    log.info(f"Checkpoint sauvegardé : {len(records)} enregistrements → {path}")

print("✅ Fonction save_checkpoint() définie")

## 🚀 Lancement du scraper

> ⚠️ Cette cellule lance le scraping complet. Peut prendre plusieurs heures pour 4000 annonces.

In [ ]:
def main():
    log.info("=== Voursa.com Real Estate Scraper v3 ===")
    log.info(f"Cible : {TARGET_ROWS} lignes | Sortie : {OUTPUT_CSV}")

    driver = build_driver(headless=True)

    try:
        # ── Phase 1 : Collecte des URLs ────────────────────────────────────
        cache_path = Path(URLS_CACHE)
        if cache_path.exists():
            log.info(f"Chargement du cache d'URLs depuis {URLS_CACHE} ...")
            items = []
            with open(cache_path, encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split("\t")
                    if len(parts) >= 1 and parts[0]:
                        items.append({
                            "url": parts[0],
                            "subcategory": parts[1] if len(parts) > 1 else ""
                        })
            log.info(f"{len(items)} URLs chargées depuis le cache")
        else:
            items = collect_urls(driver, TARGET_ROWS)

        if not items:
            log.error("Aucune URL trouvée. Abandon.")
            return

        log.info(f"Total URLs à scraper : {len(items)}")

        # ── Phase 2 : Extraction des détails ──────────────────────────────
        records:   list = []
        failed:    list = []
        seen_urls: set  = set()

        ckpt = Path(CHECKPOINT)
        if ckpt.exists():
            existing  = pd.read_csv(ckpt)
            records   = existing.to_dict("records")
            seen_urls = set(existing["url"].tolist())
            log.info(f"Reprise : {len(records)} enregistrements déjà scrapés")

        log.info("Phase 2 : Scraping des pages détail ...")
        for i, item in enumerate(items):
            if len(records) >= TARGET_ROWS:
                log.info(f"Cible de {TARGET_ROWS} lignes atteinte.")
                break

            url      = item["url"]
            sub_name = item.get("subcategory", "")

            if url in seen_urls:
                continue

            log.info(f"[{i+1}/{len(items)}] ({len(records)} sauvegardés) {url[:90]}")
            record = scrape_detail(driver, url, sub_name)

            if record:
                records.append(record)
                seen_urls.add(url)
            else:
                failed.append(url)

            if len(records) > 0 and len(records) % 100 == 0:
                save_checkpoint(records, CHECKPOINT)

        # ── Sauvegarde du CSV final ────────────────────────────────────────
        COLUMNS = [
            "titre","type_bien","type_annonce","prix",
            "surface_m2","nb_chambres","nb_salons","nb_sdb",
            "quartier","ville","description","source",
            "date_publication","caracteristiques","subcategory","url","scraped_at",
        ]
        df = pd.DataFrame(records)
        for col in COLUMNS:
            if col not in df.columns:
                df[col] = ""
        df = df[COLUMNS]
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        log.info(f"Terminé ! {len(df)} enregistrements sauvegardés dans {OUTPUT_CSV}")

        if failed:
            Path("voursa_failed_urls.txt").write_text("\n".join(failed), encoding="utf-8")
            log.info(f"{len(failed)} URLs échouées → voursa_failed_urls.txt")

    finally:
        driver.quit()
        log.info("Driver fermé.")


# Lancer le scraper
main()

## 📊 Aperçu des résultats

Exécutez cette cellule après le scraping pour explorer les données.

In [ ]:
from pathlib import Path

OUTPUT_CSV = "voursa_real_estate.csv"

if Path(OUTPUT_CSV).exists():
    df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
    print(f"✅ {len(df)} annonces chargées depuis {OUTPUT_CSV}")
    print(f"   Colonnes : {list(df.columns)}\n")
    display(df.head(10))
else:
    print("⚠️  Fichier CSV introuvable. Lancez d'abord le scraper ci-dessus.")

## 📈 Statistiques rapides

In [ ]:
if Path(OUTPUT_CSV).exists():
    df = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")

    print("── Type de bien ──────────────────────")
    print(df["type_bien"].value_counts().to_string())

    print("\n── Type d'annonce ────────────────────")
    print(df["type_annonce"].value_counts().to_string())

    print("\n── Sous-catégorie ────────────────────")
    print(df["subcategory"].value_counts().to_string())

    print("\n── Villes ────────────────────────────")
    print(df["ville"].value_counts().head(10).to_string())

    print("\n── Quartiers (top 10) ────────────────")
    print(df["quartier"].value_counts().head(10).to_string())
else:
    print("⚠️  Lancez d'abord le scraper.")